# Second case: the hypothesis Marketing will attack

**Week 1, Tuesday afternoon. Notebook 5 of 5, the second case, in pairs.** The escalated case found
the fall in behaviour inside Retail-Plus, where the same 22 members placed 26 orders against 51,
and the fall in rupees on three Business orders. Now two voices push back on the same numbers, and
the head of Retail-Plus hands you a cause. One of you answers Marketing, the other tests the cause,
and together you write the two hypotheses with the evidence that would settle each.

> **Marketing pushes back.** "A flat count can hide churn replaced by new customers, which is why we
> need acquisition. And Retail-Plus is Rs 65,250 out of a Rs 23 lakh fall; it does not matter."
>
> The marketing lead, Kalpa Retail

> **The tier offers a cause.** "It is the broken reorder button. One of my members says it has been
> broken for six weeks."
>
> The head of Retail-Plus

> **Kavya's review of the escalated case.** "The summary that lost two segments would have sent
> Meera to the wrong accounts. Tonight you will be asked what would prove your cause. Name the cause
> as a hypothesis, test its timing first, and say which data you still need."

Each step has one or more `TODO` markers. Above each `__TODOn__` placeholder is a lettered choice;
replace the placeholder with the option you pick, run the cell, then run the check under it. Run
from the top: the notebook stops at the first placeholder with a `NameError` naming it, which is
intended.

**What to post:** the nine letters in order as one string, then the two hypotheses, each with its
evidence.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit
ORDERS = kit.load_records("C2_W01_D02_orders_STUDENT.py")
SEGMENTS = ["Retail-Core", "Retail-Plus", "Business", "Student"]


def tree_for(rows):
    """The revenue tree's leaves for any list of orders, returned as a dictionary (round 3)."""
    revenue = 0
    seen = {}
    for order in rows:
        revenue += order["amount"]
        seen[order["customer_id"]] = True
    orders = len(rows)
    customers = len(seen)
    return {"revenue": revenue, "orders": orders, "customers": customers,
            "orders_per_customer": orders / customers,
            "revenue_per_order": revenue / orders}


def describe(amounts):
    """A group's typical value and spread: median, min, max and range (round 3)."""
    ordered = sorted(amounts)
    n = len(ordered)
    if n % 2 == 1:
        median = ordered[n // 2]
    else:
        median = (ordered[n // 2 - 1] + ordered[n // 2]) / 2
    return {"median": median, "min": ordered[0], "max": ordered[-1], "range": ordered[-1] - ordered[0]}


print(len(ORDERS), "orders loaded;", len(SEGMENTS), "segments")

In [ ]:
kit.side_by_side(
    kit.ladder(["Is the drop real?", "Which branch moved?", "Which segment?", "Escalated case: mix against rate", "Second case: the hypothesis"], lit=4, show=False),
    kit.vflow(["1. churn hidden in a flat count?\nthe id overlap",
               "2. too small to matter?\nthe consumer fall",
               "3. the reorder button?\nthe timing test",
               "4. the reorder button?\nthe channel test",
               "5. two hypotheses\nand the evidence for each"], show=False),
)

## Part 1. Marketing: "a flat count can hide churn replaced by new customers"

The claim is fair: 69 and 69 could be the same people or twenty lost and twenty new. The test is
the overlap of customer ids between the two quarters.

In [ ]:
q1_ids, q2_ids = set(), set()
for order in ORDERS:
    if order["quarter"] == "Q1":
        q1_ids.add(order["customer_id"])
    else:
        q2_ids.add(order["customer_id"])

# TODO 1. Which expression holds the customers lost after Q1?
#   a) q1_ids - q2_ids
#   b) q2_ids - q1_ids
#   c) q1_ids & q2_ids
#   d) len(q1_ids) - len(q2_ids)
lost = __TODO1__

# TODO 2. Which expression holds the customers new in Q2?
#   a) q1_ids & q2_ids
#   b) q1_ids - q2_ids
#   c) len(q2_ids) - len(q1_ids)
#   d) q2_ids - q1_ids
new = __TODO2__

both = q1_ids & q2_ids
kit.bars([("bought in both quarters", len(both)), ("lost after Q1", len(lost)), ("new in Q2", len(new))],
         title="Customer ids across the two quarters: nobody lost, nobody new")
kit.check("all 69 customers bought in both quarters", len(both) == 69, f"{len(both)}")
kit.check("no customer was lost and none was new", len(lost) == 0 and len(new) == 0, f"{len(lost)} lost, {len(new)} new")
kit.check("the overlap accounts for every id", len(both) + len(lost) == len(q1_ids) and len(both) + len(new) == len(q2_ids))

Write the one-line answer to Marketing's first point, with the three numbers.

## Part 2. Marketing: "Retail-Plus is Rs 65,250 out of a Rs 23 lakh fall"

The rupees are right and the comparison is wrong. Business orders run to lakhs each and consumer
orders to a few thousand, so a consumer segment's fall has to be judged against the consumer
business. Split out the three consumer segments.

In [ ]:
trees = {"Q1": {}, "Q2": {}}
for q in ("Q1", "Q2"):
    for seg in SEGMENTS:
        rows = []
        for order in ORDERS:
            if order["quarter"] == q and order["segment"] == seg:
                rows.append(order)
        trees[q][seg] = tree_for(rows)
q1, q2 = trees["Q1"], trees["Q2"]

# TODO 3. Which segments make up Kalpa's consumer business?
#   a) ["Retail-Core", "Retail-Plus", "Business"]
#   b) ["Retail-Plus"]
#   c) ["Retail-Core", "Retail-Plus", "Student"]
#   d) SEGMENTS
CONSUMER = __TODO3__

consumer_q1, consumer_q2 = 0, 0
moves = []
for seg in CONSUMER:
    consumer_q1 += q1[seg]["revenue"]
    consumer_q2 += q2[seg]["revenue"]
    moves.append((seg, q2[seg]["revenue"] - q1[seg]["revenue"]))
consumer_fall = consumer_q1 - consumer_q2
plus_fall = q1["Retail-Plus"]["revenue"] - q2["Retail-Plus"]["revenue"]
total_fall = 0
for seg in SEGMENTS:
    total_fall += q1[seg]["revenue"] - q2[seg]["revenue"]
lost_orders = 0
for seg in SEGMENTS:
    lost_orders += q1[seg]["orders"] - q2[seg]["orders"]

# TODO 4. What share of the consumer fall is Retail-Plus?
#   a) plus_fall / total_fall
#   b) plus_fall / consumer_fall
#   c) consumer_fall / total_fall
#   d) plus_fall / q1["Retail-Plus"]["revenue"]
plus_share = __TODO4__

kit.bridge(("consumer revenue, Q1", consumer_q1), moves, end_label="consumer revenue, Q2", lit=[1],
           title="The consumer business, Rs 2,28,820 to Rs 1,58,540: Retail-Plus is most of the fall")
kit.table(["Measure", "Value"],
          [("consumer revenue, Q1 to Q2", f"{kit.rupees(consumer_q1)} to {kit.rupees(consumer_q2)}"),
           ("consumer fall", f"{kit.rupees(consumer_fall)}, {consumer_fall / consumer_q1 * 100:.1f} percent"),
           ("Retail-Plus fall", f"{kit.rupees(plus_fall)}, {plus_share * 100:.0f} percent of the consumer fall"),
           ("Retail-Plus lost orders", f'{q1["Retail-Plus"]["orders"] - q2["Retail-Plus"]["orders"]} of the {lost_orders} lost')],
          caption="Marketing's second point, answered in the consumer business's own terms")
kit.check("the consumer business fell Rs 70,280", consumer_fall == 70280, kit.rupees(consumer_fall))
kit.check("Retail-Plus is 93 percent of the consumer fall", round(plus_share * 100) == 93, f"{plus_share * 100:.1f}")

Write the one-line answer to Marketing's second point, with the consumer fall, the share and the
lost orders.

**Why the lever matters as much as the size.** The frequency lever works on 22 members Kalpa has
already won. Harvard Business Review summarised studies putting the cost of acquiring a customer at
5 to 25 times the cost of retaining one (Amy Gallo, 2014, checked 29 Sep 2026); those are estimates
across industries, never Kalpa's figures, so the reply says "published estimates" and asks Finance
for Kalpa's own acquisition cost before anyone compares the two in rupees.

## Part 3. The head of Retail-Plus: "it is the broken reorder button"

A cause cannot come after its effect, so timing is the first test. Count Retail-Plus orders by
month, with Retail-Core beside it as the comparison segment.

In [ ]:
monthly = {"Retail-Plus": {}, "Retail-Core": {}}
for order in ORDERS:
    seg = order["segment"]
    if seg in monthly:
        # TODO 5. Which slice of "2026-04-22" gives the month, "2026-04"?
        #   a) [:4]
        #   b) [5:7]
        #   c) [:7]
        #   d) [-2:]
        month = order["order_date"]__TODO5__
        monthly[seg][month] = monthly[seg].get(month, 0) + 1

months = sorted(monthly["Retail-Plus"])
plus_by_month, core_by_month = [], []
for m in months:
    plus_by_month.append(monthly["Retail-Plus"][m])
    core_by_month.append(monthly["Retail-Core"].get(m, 0))
kit.table(["Month"] + months, [["Retail-Plus"] + plus_by_month, ["Retail-Core"] + core_by_month],
          caption="Orders by month")
kit.line(["Apr", "May", "Jun", "Jul", "Aug", "Sep"],
         [("Retail-Plus", plus_by_month, "bad"), ("Retail-Core", core_by_month, "plain")],
         title="Orders by month: Retail-Plus steps down in July; the complaint dates the break to late August")
kit.check("the months run April to September", months == ["2026-04", "2026-05", "2026-06", "2026-07", "2026-08", "2026-09"])
kit.check("Retail-Plus July sits below every Q1 month", plus_by_month[3] < min(plus_by_month[:3]),
          f"July {plus_by_month[3]}, Q1 lowest {min(plus_by_month[:3])}")

**The assumption, stated.** The complaint reached the head of Retail-Plus in the week of 6 October
and says six weeks; taken at its word, the break dates to 25 August. Split Q2's Retail-Plus orders
at that date, and give each side its window.

In [ ]:
from datetime import date, timedelta
BREAK = (date(2026, 10, 6) - timedelta(weeks=6)).isoformat()      # "six weeks" taken at its word

before, after = 0, 0
for order in ORDERS:
    if order["segment"] == "Retail-Plus" and order["quarter"] == "Q2":
        # TODO 6. Which condition counts the orders placed before the break?
        #   a) order["order_date"] < BREAK
        #   b) order["order_date"] > BREAK
        #   c) order["quarter"] == "Q1"
        #   d) order["order_date"][:7] == BREAK[:7]
        if __TODO6__:
            before += 1
        else:
            after += 1

days_before = (date.fromisoformat(BREAK) - date(2026, 7, 1)).days
days_after = (date(2026, 9, 30) - date.fromisoformat(BREAK)).days + 1
q1_per_week = q1["Retail-Plus"]["orders"] / 91 * 7
kit.table(["Window", "Days", "Orders", "Orders per week"],
          [("Q1, 1 Apr to 30 Jun", 91, q1["Retail-Plus"]["orders"], f"{q1_per_week:.1f}"),
           (f"Q2, 1 Jul to {date.fromisoformat(BREAK) - timedelta(days=1):%d %b}", days_before, before, f"{before / days_before * 7:.1f}"),
           (f"Q2, {date.fromisoformat(BREAK):%d %b} to 30 Sep", days_after, after, f"{after / days_after * 7:.1f}")],
          caption=f"The break, taken as {BREAK}")
kit.columns(["Q1", "Q2 before the break", "Q2 after the break"],
            [("Retail-Plus orders per week", [round(q1_per_week, 1), round(before / days_before * 7, 1), round(after / days_after * 7, 1)])],
            fmt=lambda v: f"{v:.1f}", width=560, title="Most of the fall happened before the break the complaint dates")
kit.check("the break is taken as 25 August", BREAK == "2026-08-25")
kit.check("18 Retail-Plus orders before the break and 8 after", (before, after) == (18, 8), f"{before} and {after}")

Write one line: can the broken button explain the whole fall, and what does the answer rest on?

### Worked, not a TODO: the season, a rival to both causes

July opens the monsoon quarter, so a seasonal dip would also begin in July. A season that hit every
customer would move the comparison segment too. Compare each segment's Q2 orders with its Q1 orders.

In [ ]:
plus_ratio = sum(plus_by_month[3:]) / sum(plus_by_month[:3])
core_ratio = sum(core_by_month[3:]) / sum(core_by_month[:3])
kit.columns(["Retail-Plus", "Retail-Core"],
            [("Q1 orders", [sum(plus_by_month[:3]), sum(core_by_month[:3])]),
             ("Q2 orders", [sum(plus_by_month[3:]), sum(core_by_month[3:])])],
            width=520, title="A season that hit everyone would have moved Retail-Core too")
print(f"Retail-Plus kept {plus_ratio * 100:.0f} percent of its Q1 orders; Retail-Core kept {core_ratio * 100:.0f} percent")
kit.check("Retail-Core kept more than nine tenths of its Q1 orders", core_ratio > 0.9, f"{core_ratio:.2f}")
kit.check("Retail-Plus kept about half", 0.45 < plus_ratio < 0.55, f"{plus_ratio:.2f}")

**What it says.** Retail-Core kept 95 percent of its Q1 orders and Retail-Plus 51 percent, so a season
that hit every customer does not fit. A season that hit members harder still fits, and only last
year's Q2 by segment can rule it out; it goes on the data request beside the app's logs.

## Part 4. The channel test inside Retail-Plus

A broken app feature can only stop orders placed through the app. If it were the whole cause, the
app would have fallen and the web and the store would have held.

In [ ]:
by_channel = {"Q1": {}, "Q2": {}}
for order in ORDERS:
    if order["segment"] == "Retail-Plus":
        q, ch = order["quarter"], order["channel"]
        by_channel[q][ch] = by_channel[q].get(ch, 0) + 1

CHANNELS = ["web", "store", "app"]
fell = {}
for ch in CHANNELS:
    fell[ch] = by_channel["Q2"][ch] < by_channel["Q1"][ch]

# TODO 7. If the broken reorder feature were the whole cause, which channels would have fallen?
#   a) {"web": True, "store": True, "app": True}
#   b) {"web": False, "store": False, "app": True}
#   c) {"web": True, "store": False, "app": False}
#   d) {"web": False, "store": False, "app": False}
expected_if_app_only = __TODO7__

c1, c2 = [], []
for ch in CHANNELS:
    c1.append(by_channel["Q1"][ch])
    c2.append(by_channel["Q2"][ch])
kit.columns(CHANNELS, [("Q1 orders", c1), ("Q2 orders", c2)], lit=[2],
            title="Retail-Plus orders by channel: every channel fell, the app no more than the others")
kit.check("an app-only cause leaves the app as the one channel falling",
          expected_if_app_only["app"] and sum(expected_if_app_only.values()) == 1)
kit.check("every channel fell", fell == {"web": True, "store": True, "app": True}, str(fell))
kit.check("the observed pattern differs from the app-only pattern", fell != expected_if_app_only)

Write one line: what did the channels show, and what would an app-only cause have shown?

## Part 5. Two hypotheses, and the evidence that settles each

Neither cause is proved or disproved by this export. Each is a hypothesis, and each is settled by
data Kalpa holds elsewhere.

In [ ]:
EVIDENCE = {
    "export": "orders per customer by segment from this export, again",
    "campaigns": "Marketing's new-customer campaign reach by month",
    "app_logs": "the app's reorder events and failures by week, the release that broke it, and whether members who used reorder in Q1 fell more than those who did not",
    "tier_log": "the tier's change log for benefits, prices and delivery terms, renewals, and members' support tickets",
}

# TODO 8. Which evidence settles H1, the broken reorder feature?
#   a) "export"
#   b) "campaigns"
#   c) "app_logs"
#   d) "tier_log"
settles_h1 = EVIDENCE[__TODO8__]

# TODO 9. Which evidence settles H2, something that changed for members in July?
#   a) "app_logs"
#   b) "export"
#   c) "campaigns"
#   d) "tier_log"
settles_h2 = EVIDENCE[__TODO9__]

kit.check("both hypotheses need data this export does not carry",
          EVIDENCE["export"] not in (settles_h1, settles_h2) and EVIDENCE["campaigns"] not in (settles_h1, settles_h2))
kit.check("the two hypotheses are settled by different evidence", settles_h1 != settles_h2)
kit.tree({"label": "Retail-Plus: the same 22 members, 51 orders to 26", "kind": "lit", "branches": [
    ("H1", {"label": "the broken reorder feature", "kind": "unknown", "branches": [
        ("settled by", {"label": "reorder events and failures by week; the release date", "kind": "known"})]}),
    ("H2", {"label": "a change for members in July", "kind": "unknown", "branches": [
        ("settled by", {"label": "the tier's change log, renewals, support tickets", "kind": "known"})]})]},
    title="Two hypotheses, each with the evidence that would settle it")

**Before trusting the reorder logs, ask how they are written.** A failure log that records only the
attempts that reached the server misses every member whose tap never got that far, so the members
most hurt by the break may be the ones the log cannot see. That absence is not random, which is the
morning's missing discount field in a new place: ask which system writes the log, and what it does
when the app never calls it.

### Your two hypotheses

Write each as one sentence: the cause, what the export already says about it, and the evidence that
would settle it.

*Write your hypotheses here.*

### In the interview

**[D] Marketing insists the answer is acquisition and your data says frequency; how do you make
the case in the room?**

**[SV] The customer count is flat quarter on quarter; does that prove no customers were lost?**

**[D] A stakeholder hands you a cause; how do you test it with the data you have and name the data
you need?**

Answer each aloud in under a minute, using the numbers from Parts 1 to 4.

In [ ]:
kit.check_summary()
print("Post your nine letters in order as one string, then your two hypotheses with their evidence.")